<a href="https://colab.research.google.com/github/Masked-peculiarity/CompilerDesignLab/blob/main/LEX_and_YACC_Compiler.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LEX and YACC Compiler in Colab

Drawbacks:
* Regular interrupts (Ctrl+D, Ctrl+C) for shell won't work in Colab while inputting for program.
<br>Workaround: Store your inputs in a txt file and pass it to the program.

In [1]:
#@title Install *prerqeuisites* (run this cell first to work on LEX/YACC)
!sudo apt install flex bison

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libfl-dev libfl2
Suggested packages:
  bison-doc flex-doc
The following NEW packages will be installed:
  bison flex libfl-dev libfl2
0 upgraded, 4 newly installed, 0 to remove and 2 not upgraded.
Need to get 1,072 kB of archives.
After this operation, 3,667 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 flex amd64 2.6.4-8build2 [307 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 bison amd64 2:3.8.2+dfsg-1build1 [748 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/main amd64 libfl2 amd64 2.6.4-8build2 [10.7 kB]
Get:4 http://archive.ubuntu.com/ubuntu jammy/main amd64 libfl-dev amd64 2.6.4-8build2 [6,236 B]
Fetched 1,072 kB in 1s (800 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot

## Lex only

In [ ]:
#@title Writing Lex program
%%writefile program.l

%{
    #include <stdio.h>
    int ctChar=0;
    int ctSpace=0;
    int ctWord=0;
    int ctLine=0;
%}
WORD [^ \t\n,\.:]+
EOL [\n]
BLANK [ ]
%%

{WORD} {ctWord++; ctChar+=yyleng;}
{BLANK} {ctSpace++;}
{EOL} {ctLine++;}
. {ctChar++;}
%%

void main(int argc, char *argv[]){
    if(argc!=2){
        printf("Usage:\n\t./a.out <FILENAME>\n");
        exit(0);
    }

    yyin=fopen(argv[1],"r");
    yylex();

    printf("Word Count: %d\n",ctWord);
    printf("Character Count: %d\n",ctChar);
    printf("Space Count: %d\n",ctSpace);
    printf("Line Count: %d\n",ctLine);
    fclose(yyin);

}

int yywrap(){
    return 1;
}

Overwriting program.l


if you want to use at txt as an input

In [ ]:
%%writefile program.txt

This is a sample file.

Writing program.txt


In [ ]:
#@title Shell Execution (you can rewrite the commands as per your need, eg. if you want to include a file as an input)
%%shell

lex -l program.l
gcc lex.yy.c
./a.out program.txt

Word Count: 5
Character Count: 18
Space Count: 4
Line Count: 2


## Lex and Yacc combined

In [16]:
#@title Writing YACC program
%%writefile program.y


%{
#include<stdio.h>
#include<stdlib.h>
void yyerror(const char *s) ;
int yylex() ;
%}

%token EQUAL STAR ID

%%

input : S '\n'{ printf("Valid\n") ;}
      ;

S : S EQUAL R
  | R
  ;
L : STAR R
  | ID
  ;
R : L
  ;

%%

void yyerror(const char *s) {
printf("Invalid\n") ;
}
int main() {
yyparse();
return 0 ;;
}


Overwriting program.y


In [15]:
#@title Writing Lex program
%%writefile program.l
%{
#include "y.tab.h"
%}

%%

\= { return EQUAL ;}
\* { return STAR ;}
id { return ID ;}
[ \t] ;
\n { return '\n' ;}
. { return yytext[0] ;}

%%
int yywrap() {
return 1 ;
}

Overwriting program.l


if you want to use at txt as an input

In [4]:
%%writefile program.txt
%%bash
cat > input.txt << EOF
id=id
*id=id
id=*id
EOF
This is a sample file.

Writing program.txt


In [24]:
#@title Shell Execution
%%shell

yacc -d program.y
lex program.l
cc y.tab.c lex.yy.c -ll -o parser

cat > input.txt << EOF
*id=
EOF

./parser < input.txt

Invalid
